# B1259 component RM reproducibility
Run the settings, definitions and synthetic control first. The last two cells run the original component analysis and the PSRSALSA comparison independently. Set the archive path in the original settings cell. The synthetic control needs NumPy, Matplotlib and a C compiler accessible as `cc`. Archive processing also needs PSRCHIVE.

The original estimator coherently sums polarisation across longitude before taking its magnitude. The comparison takes the magnitude in each longitude bin first. Opposite polarisation vectors can therefore cancel in the original spectrum while adding in the collapsed spectrum.

Implementation references for commit `2efbaf4592e08fdb76002bbc9dc25e8e4441813b` (22 February 2022) include [prog/rmsynth.c](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/prog/rmsynth.c), [lib/rmsynth.c](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/lib/rmsynth.c), [lib/psrio.c](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/lib/psrio.c), [lib/psrio_paswing.c](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/lib/psrio_paswing.c), [lib/amoeba_nmsimplex_d.c](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/lib/amoeba_nmsimplex_d.c).

# Component RM synthesis for folded pulsar archives
Set the input below and run all cells in a Python kernel with PSRCHIVE, NumPy and Matplotlib. Each archive gets a frequency-averaged intensity profile, four frequency–pulse-phase maps averaged over subintegrations, and one Faraday spectrum per detected component. Use calibrated full-Stokes data. Figures appear inline and the final table is saved as CSV.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

#INPUT = Path("/fred/oz341/B1259_kavya/parkes/P116/DM_RM_Updated_v4_extracted_only/rmsynth_for_all/low_res_imp_epochs/")    # Directory or one archive file
INPUT = Path("/fred/oz341/B1259_kavya/parkes/high_res/test_archives_rfi/sub/ext/") #high_res
EXTENSION = ".center"              # Directory mode only, e.g. ".ar.clfd.T"
OUTPUT = Path("files_close_to_peri_2e4")
FREQ_MHZ = (None, None)       # Full band, or e.g. (2600, 4032)
RM_LIMITS = (-10000., 10000.)   # Total RM in rad/m²
GRID_POINTS = 20000
DRAWS = 500
SEED = 42
PEAK_SIGMA = 3.0              # Minimum smoothed intensity above baseline
SMOOTH_PHASE = 0.004          # Gaussian smoothing width in rotations

## Read and align the archive
Dedisperse at the header DM and average subintegrations with PSRCHIVE. Retain native phase and frequency resolution. Flagged and invalid channels appear blank in the maps. As in the source notebook, usable channels contribute equally.

In [ ]:
def read_archive(path):
    import psrchive
    ar = psrchive.Archive_load(str(path))
    if ar.get_npol() != 4:
        raise ValueError("Four polarisation products are required")
    ar.convert_state("Stokes")
    ar.dedisperse()
    ar.tscrunch()
    integ = ar.get_Integration(0)
    freq = np.array([integ.get_centre_frequency(i) for i in range(ar.get_nchan())])
    cube = ar.get_data()[0].astype(float)
    weights = ar.get_weights()[0]
    band = np.isfinite(freq) & (freq > 0)
    lo, hi = FREQ_MHZ
    if lo is not None:
        band &= freq >= lo
    if hi is not None:
        band &= freq <= hi
    order = np.flatnonzero(band)[np.argsort(freq[band])]
    freq, cube = freq[order], cube[:, order]
    valid = (np.isfinite(weights[order]) & (weights[order] > 0)
             & np.isfinite(cube).all(axis=(0, 2)))
    if valid.sum() < 4 or np.ptp(freq[valid]) == 0:
        raise ValueError("Insufficient usable frequency coverage")
    cube[:, ~valid] = np.nan
    applied = float(ar.get_rotation_measure()) if ar.get_faraday_corrected() else 0.
    return freq, cube, valid, applied

## Identify pulse components
Smooth the band-averaged I profile with periodic boundaries. Rotate the search origin to the intensity minimum so a peak crossing phase zero stays intact. Require significance and prominence, then keep up to two peaks. A gate extends to the two-sigma baseline crossing or the valley between peaks. Leading and trailing follow peak order in the original archive phase. Inspect the shaded gates before interpreting a measurement.

In [ ]:
def detect(profile):
    n = len(profile)
    width = max(0.7, n * SMOOTH_PHASE)
    radius = max(1, round(4*width))
    kernel = np.exp(-0.5*(np.arange(-radius, radius+1)/width)**2)
    smooth = np.convolve(np.pad(profile, radius, mode="wrap"), kernel/kernel.sum(), mode="valid")
    origin = int(smooth.argmin())
    y = np.roll(smooth, -origin)
    baseline = np.median(y)
    noise = max(1.4826 * np.median(abs(y - baseline)), np.finfo(float).tiny)
    candidates = np.flatnonzero((y > np.roll(y, 1)) & (y >= np.roll(y, -1))
                                & (y > baseline + PEAK_SIGMA*noise))
    selected = []
    for peak in candidates[np.argsort(y[candidates])[::-1]]:
        if not selected:
            selected.append(peak)
            continue
        a, b = sorted([selected[0], peak])
        # A separate component needs a resolved valley on both circular paths.
        saddle = max(y[a:b+1].min(), min(y[:a+1].min(), y[b:].min()))
        if min(b-a, n-b+a) >= max(2, round(3*width)) and y[peak]-saddle >= PEAK_SIGMA*noise:
            selected.append(peak)
            break
    chosen = sorted(selected)
    gates = []
    for p in chosen:
        left, right = p, p
        while left > 0 and y[left-1] > baseline + 2*noise:
            left -= 1
        while right < n-1 and y[right+1] > baseline + 2*noise:
            right += 1
        if len(chosen) == 2:
            a, b = chosen
            valley = a + np.argmin(y[a:b+1])
            if p == a:
                right = min(right, valley)
            else:
                left = max(left, valley+1)
        gates.append(np.roll((np.arange(n) >= left) & (np.arange(n) <= right), origin))
    # Exclude other significant emission and pad the noise window on both sides.
    emission = np.roll(y > baseline + 2*noise, origin)
    pad = max(1, round(2*width))
    padded = np.pad(emission, pad, mode="wrap")
    off = np.convolve(padded.astype(float), np.ones(2*pad+1), mode="valid") == 0
    gates.sort(key=lambda gate: np.argmax(np.where(gate, smooth, -np.inf)))
    return gates, off

## Synthesize one RM per gate
Subtract the channel-wise off-pulse baseline, sum Q + iU across the gate, and maximize the modulus of its derotated channel mean. Trial values denote **total RM**. Only the difference from any RM already removed by PSRCHIVE enters the rotation. A three-point parabola refines interior peaks.

For a statistical 1σ error, perturb the observed gate sum with resampled, centred off-pulse Q/U spectra and repeat the full search. Sampling whole spectra preserves simultaneous channel and Q/U noise correlations. Independent phase-bin noise and a representative off-pulse window are assumed. The scaling includes uncertainty in the subtracted baseline. Inspect broad or competing peaks and widen the grid for boundary hits.

In [ ]:
def synthesize(freq, cube, gate, off, applied, rng):
    pol = cube[1] + 1j*cube[2]
    quiet = pol[:, off]
    quiet = (quiet - quiet.mean(axis=1, keepdims=True)) * np.sqrt(off.sum()/(off.sum()-1))
    signal = pol[:, gate].sum(axis=1)
    n = gate.sum()
    samples = np.empty((len(freq), DRAWS+1), complex)
    samples[:, 0] = signal
    for j in range(1, DRAWS+1):
        idx = rng.integers(off.sum(), size=n)
        samples[:, j] = signal + quiet[:, idx].sum(axis=1)*np.sqrt(1+n/off.sum())
    lam2 = (299792458 / (freq*1e6))**2
    grid = np.linspace(*RM_LIMITS, GRID_POINTS)
    power = np.empty((len(grid), DRAWS+1))
    for i in range(0, len(grid), 128):
        trial = grid[i:i+128, None] - applied
        kernel = np.exp(-2j*trial*(lam2-lam2.mean()))
        power[i:i+128] = abs(kernel @ samples)/len(freq)
    col = np.arange(DRAWS+1)
    peak = power.argmax(axis=0)
    boundary = (peak == 0) | (peak == len(grid)-1)
    k = peak.clip(1, len(grid)-2)
    a,b,c = power[k-1,col], power[k,col], power[k+1,col]
    offset = np.divide(0.5*(a-c), a-2*b+c, out=np.zeros_like(a), where=(a-2*b+c != 0))
    rm = grid[peak] + np.where(boundary, 0., offset)*(grid[1]-grid[0])
    return grid, power[:,0], rm[0], rm[1:].std(ddof=1), boundary

## Process the selected files
The profile marks gate bins in colour and off-pulse bins in grey along its lower edge. Each map uses its own symmetric colour scale in archive intensity units. Low-significance profiles still get diagnostic figures. The CSV records gate indices in original phase order, total RM, thermal error estimates and search-boundary flags.

In [ ]:
def process(path, rng):
    freq, cube, valid, applied = read_archive(path)
    gates, off = detect(cube[0, valid].mean(axis=0))
    phase = np.arange(cube.shape[-1])/cube.shape[-1]
    labels = ["single"] if len(gates) == 1 else ["leading", "trailing"]
    enough_noise = off.sum() >= 8
    if enough_noise:
        cube -= cube[:, :, off].mean(axis=2, keepdims=True)
    fig, axes = plt.subplots(5, 1, sharex=True, figsize=(10, 11), constrained_layout=True)
    ax = axes[0]
    ax.plot(phase, cube[0, valid].mean(axis=0), color="black", lw=1)
    ax.plot(phase[off], np.full(off.sum(), .04), "|", color=".65",
            transform=ax.get_xaxis_transform(), label="off-pulse")
    for j, (label, gate) in enumerate(zip(labels, gates)):
        ax.fill_between(phase, 0, 1, where=gate, step="mid", color=f"C{j}",
                        alpha=.25, transform=ax.get_xaxis_transform(), label=label)
    ax.legend()
    ax.set(title=path.name, ylabel="Mean I")
    for panel, data, stokes in zip(axes[1:], cube, "IQUV"):
        lim = np.nanpercentile(abs(data), 99) or 1.
        im = panel.pcolormesh(phase, freq, data, shading="nearest", cmap="RdBu_r",
                              vmin=-lim, vmax=lim, rasterized=True)
        panel.set_ylabel(f"{stokes}  ·  MHz")
        fig.colorbar(im, ax=panel)
    axes[-1].set_xlabel("Pulse phase (rotations)")
    fig.savefig(OUTPUT / (path.name + ".stokes.png"), dpi=140)
    plt.show()
    plt.close(fig)
    if not gates or not enough_noise:
        return [dict(archive=path.name, status="no significant peak" if not gates else "too few off-pulse bins")]
    rows = []
    fig, axes = plt.subplots(len(gates), 1, squeeze=False, figsize=(9, 3*len(gates)),
                             constrained_layout=True)
    for ax, label, gate in zip(axes[:, 0], labels, gates):
        grid, power, rm, err, edge = synthesize(freq[valid], cube[:,valid], gate, off, applied, rng)
        status = "review search limits" if edge.any() else "ok"
        ax.plot(grid, power)
        ax.axvline(rm, color="C1", ls="--")
        ax.set(title=f"{path.name} · {label}: {rm:.2f} ± {err:.2f} rad/m² ({status})",
               xlabel="Total RM (rad/m²)", ylabel="Polarised amplitude")
        rows.append(dict(archive=path.name, component=label, rm=rm, sigma_rm=err,
                         applied_rm=applied, channels=valid.sum(), off_bins=off.sum(),
                         gate_bins=" ".join(map(str, np.flatnonzero(gate))),
                         peak_at_edge=edge[0], draw_edge_fraction=edge[1:].mean(), status=status))
    fig.savefig(OUTPUT / (path.name + ".rm.png"), dpi=140)
    plt.show()
    plt.close(fig)
    return rows

## Comparison settings
Use zero-based inclusive bin pairs in the original archive phase. A reversed pair wraps through bin zero. Each dictionary entry is one independent `-onpulse` run. To collapse both components together, put both intervals under one name. `ONPULSE2` selects the signal window for the analytic error and its complement supplies the noise. Record the same two windows on OzSTAR. Automatic detection is a convenience when no explicit windows are supplied.

In [ ]:
REGIONS = None  # e.g. {"leading": [(80, 91)], "trailing": [(104, 115)]}
ONPULSE2 = None  # e.g. [(78, 118)]; default = union of selected regions
PS_GRID = (*RM_LIMITS, GRID_POINTS)  # -rm "min max count", inclusive
PS_DRAWS = 0  # -bootstrap N; set >= 2 for an uncertainty, e.g. 500
PS_LONG = False  # also fit each selected bin, equivalent to an additional -long run
FTOL = 0.001
SUBTRACT_BASELINE = False  # default assumes the input already has its baseline removed
RUN_ORIGINAL = True
RUN_COMPARISON = True

## Instrumental response and fit
The model is offset + scale × the modulus of the mean channel phasor at trial RM minus fitted RM. All grid points enter the unweighted squared residual. The initial RM is the grid midpoint, so a broad search can converge to a sidelobe. Compare the fitted curve with the sampled peak. The next cell builds the pinned upstream double-precision simplex routine in a temporary directory. Its original licence is included.

In [ ]:
import ctypes as ct
import subprocess, tempfile
from numpy.ctypeslib import ndpointer

solver_source = '/* \n * Program: nmsimplex.c\n * Author : Michael F. Hutt\n * http://www.mikehutt.com\n * 11/3/97\n *\n * An implementation of the Nelder-Mead simplex method.\n *\n * Copyright (c) 1997-2011 <Michael F. Hutt>\n *\n * Permission is hereby granted, free of charge, to any person obtaining\n * a copy of this software and associated documentation files (the\n * "Software"), to deal in the Software without restriction, including\n * without limitation the rights to use, copy, modify, merge, publish,\n * distribute, sublicense, and/or sell copies of the Software, and to\n * permit persons to whom the Software is furnished to do so, subject to\n * the following conditions:\n *\n * The above copyright notice and this permission notice shall be\n * included in all copies or substantial portions of the Software.\n *\n * THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,\n * EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF\n * MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND\n * NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE\n * LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION\n * OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION\n * WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n *\n *\n * Jan. 6, 1999 \n * Modified to conform to the algorithm presented\n * in Margaret H. Wright\'s paper on Direct Search Methods.\n *\n * Jul. 23, 2007\n * Fixed memory leak.\n *\n * Mar. 1, 2011\n * Added constraints.\n *\n * Patrick Weltevrede added the following functionality:\n * A step size should be provided, rather than a scale\n * A verbose option is added and output format is different\n * Removed the constrain option\n * Added the reachedEpsilon parameter\n * Changed convergence condition\n */\n\n/* This is an exact copy of amoeba_nmsimplex.c (float version) to make\n   the double version. This file should be regeneratable by\n   applying the following substitutions:\n\n   float     ->  double \n   _nmsimplex -> _nmsimplex_d\n   %f        -> %lf\n*/\n\n\n#include <stdlib.h>\n#include <math.h>\n#include <stdio.h>\n\n#define MAX_IT      5000      /* maximum number of iterations */\n#define ALPHA       1.0       /* reflection coefficient */\n#define BETA        0.5       /* contraction coefficient */\n#define GAMMA       2.0       /* expansion coefficient */\n\ndouble amoeba_nmsimplex_d(double (*objfunc)(double[]), double start[], double dx[], int n, double EPSILON, int *nfunc, double *reachedEpsilon, int verbose)\n{\n  //  void (*constrain)(double[],int n);\n\n  int vs;         /* vertex with smallest value */\n  int vh;         /* vertex with next smallest value */\n  int vg;         /* vertex with largest value */\n  \n  int i,j,m,row;\n  int k;   \t      /* track the number of function evaluations */\n  \n  double **v;     /* holds vertices of simplex */\n  //\tdouble pn,qn;   /* values used to create initial simplex */\n  double *f;      /* value of function at each vertex */\n  double fr;      /* value of function at reflection point */\n  double fe;      /* value of function at expansion point */\n  double fc;      /* value of function at contraction point */\n  double *vr;     /* reflection - coordinates */\n  double *ve;     /* expansion - coordinates */\n  double *vc;     /* contraction - coordinates */\n  double *vm;     /* centroid - coordinates */\n  double min;\n\t\n  double s,cent;\n  //  double fsum,favg;\n\n  /* dynamically allocate arrays */\n  \n  /* allocate the rows of the arrays */\n  v =  (double **) malloc ((n+1) * sizeof(double *));\n  f =  (double *) malloc ((n+1) * sizeof(double));\n  vr = (double *) malloc (n * sizeof(double));\n  ve = (double *) malloc (n * sizeof(double));  \n  vc = (double *) malloc (n * sizeof(double));  \n  vm = (double *) malloc (n * sizeof(double));  \n\t\n  /* allocate the columns of the arrays */\n  for (i=0;i<=n;i++) {\n    v[i] = (double *) malloc (n * sizeof(double));\n  }\n  \n  /* create the initial simplex */\n  /* assume one of the vertices is 0,0 */\n  /*\t\n\tpn = scale*(sqrt(n+1)-1+n)/(n*sqrt(2));\n\tqn = scale*(sqrt(n+1)-1)/(n*sqrt(2));\n  */\n  \n  for (i=0;i<n;i++) {\n    v[0][i] = start[i];\n  }\n  \n  for (i=1;i<=n;i++) {\n    for (j=0;j<n;j++) {\n      if (i-1 == j) {\n\t//\t\t\t\tv[i][j] = pn + start[j];\n\tv[i][j] = start[j]+dx[j];\n      }\n      else {\n\t//\t\t\t\tv[i][j] = qn + start[j];\n\tv[i][j] = start[j];\n      }\n    }\n  }\n  \n  //\tif (constrain != NULL) {\n  //    constrain(v[j],n);\n  //  } \n  \n  /* find the initial function values */\n  for (j=0;j<=n;j++) {\n    f[j] = objfunc(v[j]);\n  }\n  \n  k = n+1;\n  \n  /* print out the initial values */\n  if(verbose) {\n    printf("Initial Values\\n");\n    for (j=0;j<=n;j++) {\n      printf("Vertex %d:", j+1);\n      for (i=0;i<n;i++) {\n\tprintf(" %lf",v[j][i]);\n      }\n      printf(": %lf\\n",f[j]);\n    }\n  }\n\t\n\t\n  /* begin the main loop of the minimization */\n  int nritterations;\n  for (nritterations=1;nritterations<=MAX_IT;(nritterations)++) {     \n    /* find the index of the largest value */\n    vg=0;\n    for (j=0;j<=n;j++) {\n      if (f[j] > f[vg]) {\n\tvg = j;\n      }\n    }\n    \n    /* find the index of the smallest value */\n    vs=0;\n    for (j=0;j<=n;j++) {\n      if (f[j] < f[vs]) {\n\tvs = j;\n      }\n    }\n    \n    /* find the index of the second largest value */\n    vh=vs;\n    for (j=0;j<=n;j++) {\n      if (f[j] > f[vh] && f[j] < f[vg]) {\n\tvh = j;\n      }\n    }\n    \n    /* calculate the centroid */\n    for (j=0;j<=n-1;j++) {\n      cent=0.0;\n      for (m=0;m<=n;m++) {\n\tif (m!=vg) {\n\t  cent += v[m][j];\n\t}\n      }\n      vm[j] = cent/n;\n    }\n    \n    /* reflect vg to new vertex vr */\n    for (j=0;j<=n-1;j++) {\n      /*vr[j] = (1+ALPHA)*vm[j] - ALPHA*v[vg][j];*/\n      vr[j] = vm[j]+ALPHA*(vm[j]-v[vg][j]);\n    }\n    //\t\tif (constrain != NULL) {\n    //      constrain(vr,n);\n    //    }\n    fr = objfunc(vr);\n    k++;\n    \n    if (fr < f[vh] && fr >= f[vs]) {\n      for (j=0;j<=n-1;j++) {\n\tv[vg][j] = vr[j];\n      }\n      f[vg] = fr;\n    }\n    \n    /* investigate a step further in this direction */\n    if ( fr <  f[vs]) {\n      for (j=0;j<=n-1;j++) {\n\t/*ve[j] = GAMMA*vr[j] + (1-GAMMA)*vm[j];*/\n\tve[j] = vm[j]+GAMMA*(vr[j]-vm[j]);\n      }\n      //\t\t\tif (constrain != NULL) {\n      //        constrain(ve,n);\n      //      }\n      fe = objfunc(ve);\n      k++;\n      \n      /* by making fe < fr as opposed to fe < f[vs], \t\t\t   \n\t Rosenbrocks function takes 63 iterations as opposed \n\t to 64 when using double variables. */\n      \n      if (fe < fr) {\n\tfor (j=0;j<=n-1;j++) {\n\t  v[vg][j] = ve[j];\n\t}\n\tf[vg] = fe;\n      }\n      else {\n\tfor (j=0;j<=n-1;j++) {\n\t  v[vg][j] = vr[j];\n\t}\n\tf[vg] = fr;\n      }\n    }\n    \n    /* check to see if a contraction is necessary */\n    if (fr >= f[vh]) {\n      if (fr < f[vg] && fr >= f[vh]) {\n\t/* perform outside contraction */\n\tfor (j=0;j<=n-1;j++) {\n\t  /*vc[j] = BETA*v[vg][j] + (1-BETA)*vm[j];*/\n\t  vc[j] = vm[j]+BETA*(vr[j]-vm[j]);\n\t}\n\t//\t\t\t\tif (constrain != NULL) {\n\t//          constrain(vc,n);\n\t//        }\n\tfc = objfunc(vc);\n\tk++;\n      }\n      else {\n\t/* perform inside contraction */\n\tfor (j=0;j<=n-1;j++) {\n\t  /*vc[j] = BETA*v[vg][j] + (1-BETA)*vm[j];*/\n\t  vc[j] = vm[j]-BETA*(vm[j]-v[vg][j]);\n\t}\n\t//\t\t\t\tif (constrain != NULL) {\n\t//          constrain(vc,n);\n\t//        }\n\tfc = objfunc(vc);\n\tk++;\n      }\n      \n      \n      if (fc < f[vg]) {\n\tfor (j=0;j<=n-1;j++) {\n\t  v[vg][j] = vc[j];\n\t}\n\tf[vg] = fc;\n      }\n      /* at this point the contraction is not successful,\n\t we must halve the distance from vs to all the \n\t vertices of the simplex and then continue.\n\t 10/31/97 - modified to account for ALL vertices. \n      */\n      else {\n\tfor (row=0;row<=n;row++) {\n\t  if (row != vs) {\n\t    for (j=0;j<=n-1;j++) {\n\t      v[row][j] = v[vs][j]+(v[row][j]-v[vs][j])/2.0;\n\t    }\n\t  }\n\t}\n\t//\t\t\t\tif (constrain != NULL) {\n\t//          constrain(v[vg],n);\n\t//        }\n\tf[vg] = objfunc(v[vg]);\n\tk++;\n\t//\t\t\t\tif (constrain != NULL) {\n\t//          constrain(v[vh],n);\n\t//        }\n\tf[vh] = objfunc(v[vh]);\n\tk++;\n\t\n\t\n      }\n    }\n    \n    /* print out the value at each iteration */\n    if(verbose) {\n      printf("Iteration %d\\n",nritterations);\n      for (j=0;j<=n;j++) {\n\tprintf("Vertex %d:", j+1);\n\tfor (i=0;i<n;i++) {\n\t  printf(" %lf",v[j][i]);\n\t}\n\tprintf(": %lf\\n",f[j]);\n      }\n    }\n    \n    /* test for convergence */\n    /*\n    fsum = 0.0;\n    for (j=0;j<=n;j++) {\n      fsum += f[j];\n    }\n    favg = fsum/(n+1);\n    s = 0.0;\n    for (j=0;j<=n;j++) {\n      s += pow((f[j]-favg),2.0)/(n);\n    }\n    s = sqrt(s);*/\n    double yhi, ylo;\n\n    yhi = ylo = f[0];\n    for (j=0;j<=n;j++) {\n      if(f[j] > yhi)\n\tyhi = f[j];\n      if(f[j] < ylo)\n\tylo = f[j];\n    }\n    s = 2*fabs(yhi-ylo)/(fabs(yhi)+fabs(ylo));\n\n    if (s < EPSILON) break;\n  }\n  /* end main loop of the minimization */\n  \n  *reachedEpsilon = s;\n\n  /* find the index of the smallest value */\n  vs=0;\n  for (j=0;j<=n;j++) {\n    if (f[j] < f[vs]) {\n      vs = j;\n    }\n  }\n  \n  if(verbose) {\n    printf("The minimum was found at epsilon=%e:\\n  ", *reachedEpsilon); \n    for (j=0;j<n;j++) {\n      printf(" %e",v[vs][j]);\n    }\n    printf("\\n");\n  }\n  for (j=0;j<n;j++) {\n    start[j] = v[vs][j];\n  }\n  min=objfunc(v[vs]);\n  k++;\n  if(verbose) {\n    printf("%d Function Evaluations\\n",k);\n    printf("%d Iterations through program\\n",nritterations);\n  }\n  *nfunc = k;\n\n  free(f);\n  free(vr);\n  free(ve);\n  free(vc);\n  free(vm);\n  for (i=0;i<=n;i++) {\n    free (v[i]);\n  }\n  free(v);\n  return min;\n}\n'
build = tempfile.TemporaryDirectory()
src = Path(build.name) / "simplex.c"
src.write_text(solver_source)
libfile = src.with_suffix(".so")
subprocess.run(["cc", "-O2", "-shared", "-fPIC", str(src), "-lm", "-o", str(libfile)], check=True)
library = ct.CDLL(str(libfile))
objective_type = ct.CFUNCTYPE(ct.c_double, ct.POINTER(ct.c_double))
simplex = library.amoeba_nmsimplex_d
array_type = ndpointer(dtype=np.float64, ndim=1, flags="C_CONTIGUOUS")
simplex.argtypes = [objective_type, array_type, array_type, ct.c_int, ct.c_double,
                    ct.POINTER(ct.c_int), ct.POINTER(ct.c_double), ct.c_int]
simplex.restype = ct.c_double

In [ ]:
def fit_response(grid, spectrum, nominal_freq):
    # Factor the shift out of the RM grid so repeated evaluations reuse phasors.
    l2 = (299.792458 / nominal_freq)**2
    bank = np.exp(-2j * grid[:, None] * l2)
    def response(shift):
        return abs(bank @ np.exp(2j * shift * l2)) / len(l2)
    @objective_type
    def loss(x):
        residual = x[1] * response(x[0]) + x[2] - spectrum
        return float(residual @ residual)
    peak = float(spectrum.max())
    if peak <= 0 or not np.isfinite(spectrum).all():
        raise ValueError("Cannot fit an empty or nonfinite spectrum")
    params = np.array([(grid[0]+grid[-1])/2, peak, 0.])
    step = .05*np.array([np.ptp(grid), peak, peak])
    reached = ct.c_double()
    simplex(loss, params, step, 3, FTOL, ct.byref(ct.c_int()), ct.byref(reached), 0)
    if not np.isfinite(reached.value) or reached.value > FTOL:
        raise RuntimeError("Instrumental response fit failed to converge")
    shift, scale, offset = params
    return params, scale*response(shift) + offset

def mask_regions(pairs, n):
    bins = np.arange(n)
    mask = np.zeros(n, bool)
    for left, right in pairs:
        if not (0 <= left < n and 0 <= right < n):
            raise ValueError(f"Region {(left,right)} outside 0..{n-1}")
        mask |= ((bins >= left) & (bins <= right) if left <= right
                 else (bins >= left) | (bins <= right))
    if not mask.any():
        raise ValueError("Empty region")
    return mask

def analytic_error(pol, freq, nominal, on2, rm, longitude=False):
    derotated = np.exp(-2j * rm * (299.792458/freq)**2) @ pol
    raw_l2 = abs(derotated)**2
    rms_l = np.sqrt(raw_l2[~on2].mean())
    debiased = np.sqrt(np.maximum(raw_l2 - rms_l**2/2, 0))
    total = debiased[on2].sum()
    count = on2.sum()
    sn = total/(np.sqrt(count)*rms_l) if rms_l > 0 else np.nan
    if longitude:
        sn /= count
    width = 1/np.std((299.792458/nominal)**2, ddof=1)
    return dict(sigma_analytic=width/(2*sn) if sn > 0 else np.nan,
                fwhm=width, sn_l=sn, rms_l=rms_l, sum_l=total)

def compare_cube(freq, nominal, cube, gate, on2, settings, draws, long_mode, seed):
    low, high, count = settings
    if low >= high or count < 3 or draws < 0 or draws == 1 or FTOL <= 0:
        raise ValueError("Check RM grid, bootstrap count and fit tolerance")
    if not gate.any() or not on2.any() or (~on2).sum() < 2:
        raise ValueError("Need signal bins and at least two noise bins")
    grid = np.linspace(low, high, count)
    pol = cube[1] + 1j*cube[2]
    signal = pol[:, gate]
    rms_i = cube[0][:, ~on2].std(axis=1, ddof=0)[:, None]
    wavelength = (299.792458/freq).astype(np.float32)
    phase = -2*grid.astype(np.float32)[:, None]*(wavelength*wavelength)
    kernel = np.exp(1j*phase).astype(np.complex64)
    rng = np.random.default_rng(seed)
    bins = np.flatnonzero(gate)
    estimates, rows = [], []
    for iteration in range(draws+1):
        sample = signal
        if iteration:
            noise = rng.normal(size=signal.shape) + 1j*rng.normal(size=signal.shape)
            sample = signal + rms_i*noise
        amps = abs(kernel @ sample.astype(np.complex64))
        collapsed = amps.sum(axis=1)
        targets = [collapsed] + (list(amps.T) if long_mode else [])
        fits = [fit_response(grid, y, nominal) for y in targets]
        rmlist = [params[0] for params, curve in fits]
        if iteration:
            estimates.append(rmlist)
            continue
        products = dict(grid=grid, bins=bins, amplitudes=amps, collapsed=collapsed,
                        fitted_curve=fits[0][1], onpulse2=np.flatnonzero(on2),
                        freq_mhz=freq, nominal_mhz=nominal, channel_sigma_i=rms_i[:,0])
        for j, ((rm, scale, offset), _) in enumerate(fits):
            err = analytic_error(pol, freq, nominal, on2,
                                 np.mean(rmlist[1:]) if j else rm, bool(j))
            rows.append(dict(bin=int(bins[j-1]) if j else "collapsed", rm=rm,
                             scale=scale, offset=offset, grid_peak=grid[targets[j].argmax()],
                             outside_grid=not low <= rm <= high, **err))
    samples = np.asarray(estimates).reshape(draws,len(rows))
    products["bootstrap_rm"] = samples
    for j,row in enumerate(rows):
        row["sigma_bootstrap"] = samples[:,j].std(ddof=0) if draws else np.nan
    return rows, products

## Synthetic reproducibility control
Two bins have opposite Q/U and the same injected RM. The coherent gate sum vanishes while the collapsed spectrum has twice the single-bin amplitude. The assertions also check the RM sign, fitted response, bootstrap convention and fixed-seed repeatability.

In [ ]:
def synthetic_check():
    freq = np.linspace(900., 1500., 12)
    cube = np.random.default_rng(19).normal(0,.02,(4,12,32))
    truth = 7.5
    phasor = np.exp(2j*truth*(299.792458/freq)**2)
    cube[1,:,8], cube[2,:,8] = phasor.real, phasor.imag
    cube[1,:,9], cube[2,:,9] = -phasor.real, -phasor.imag
    gate = mask_regions([(8,9)],32)
    args = (freq,freq,cube,gate,gate,(-30.,30.,121),4,True,31)
    rows, product = compare_cube(*args)
    # Independent scalar channel sum across the entire grid, including sidelobes.
    expected = [2*abs(sum(np.exp(2j*(truth-r)*(299.792458/freq)**2)))
                for r in product["grid"]]
    np.testing.assert_allclose(product["collapsed"], expected, atol=1e-5)
    np.testing.assert_allclose(product["fitted_curve"], expected, atol=.02)
    np.testing.assert_allclose(rows[1]["sigma_analytic"], rows[2]["sigma_analytic"])
    repeat = compare_cube(*args)[1]
    np.testing.assert_array_equal(product["bootstrap_rm"], repeat["bootstrap_rm"])
    np.testing.assert_allclose(product["collapsed"].max(),24,rtol=1e-6)
    np.testing.assert_allclose((cube[1]+1j*cube[2])[:,gate].sum(axis=1),0)
    np.testing.assert_allclose([r["rm"] for r in rows],truth,atol=.01)
    np.testing.assert_allclose(rows[0]["sigma_bootstrap"],product["bootstrap_rm"][:,0].std())
    assert rows[0]["sigma_analytic"] > 0 and rows[0]["sigma_bootstrap"] > 0
    print(f"Injected {truth:.3f}, recovered {rows[0]['rm']:.5f} rad/m²; all checks passed.")
synthetic_check()

## Archive comparison
Use the same dedispersed, time-integrated, baseline-subtracted Stokes data in both programs. The comparison rejects Faraday-corrected archives. The legacy reader sorts by frequency and filters by weight. Here invalid channels are represented by zeros on the full selected grid, including in the response geometry. Verify C reads the same samples and flags. PSRCHIVE dedispersion, subintegration weighting, float reductions and channel order can differ. For nonuniform labels supply the nominal, nonweighted channel centres from the C header separately by setting the first centre and increment below (ascending order after sorting).

The bootstrap perturbs observed Q and U independently using the centred off-pulse I population standard deviation in each channel. C also draws unused I and V noise. NumPy and GSL generate different realisations. Compare distributions with the same number of draws. The reported spread uses divisor N. For the analytic error the code uses raw off-pulse RMS(L), debiases each L in quadrature and evaluates the width as the reciprocal sample standard deviation of nominal λ². In longitude mode the mean bin RM sets the derotation and all bins share one analytic error.

The fit uses the upstream optimiser and tolerance. Complex matrix multiplication and factorisation of the response change rounding. This notebook implements the default instrumental fit. The original notebook’s three-point peak refinement differs from the optional C whole-grid parabola. Memory use scales with grid size times channel count and selected bins. Start with a single archive and the same narrow RM grid used for the C comparison.

In [ ]:
NOMINAL_FIRST_STEP = None  # (first MHz, positive channel spacing MHz)

def run_ps(path):
    import psrchive
    if psrchive.Archive_load(str(path)).get_faraday_corrected():
        raise ValueError("Comparison requires an archive before Faraday correction")
    freq,cube,valid,_ = read_archive(path)
    if REGIONS is None:
        gates,_ = detect(cube[0,valid].mean(axis=0))
        labels = ["single"] if len(gates)==1 else ["leading","trailing"]
        regions = dict(zip(labels,gates))
    else:
        regions = {key:mask_regions(value,cube.shape[-1]) for key,value in REGIONS.items()}
    if not regions:
        raise ValueError("No gates detected; set REGIONS explicitly")
    on2 = (mask_regions(ONPULSE2,cube.shape[-1]) if ONPULSE2 is not None
           else np.logical_or.reduce(list(regions.values())))
    cube[:,~valid] = 0
    if SUBTRACT_BASELINE:
        cube -= cube[:,:,~on2].mean(axis=2,keepdims=True)
    if NOMINAL_FIRST_STEP is None:
        if not np.allclose(np.diff(freq), np.median(np.diff(freq)),rtol=1e-4):
            raise ValueError("Nonuniform labels: set NOMINAL_FIRST_STEP from the header")
        nominal = np.linspace(freq[0],freq[-1],len(freq))
    else:
        first,step = NOMINAL_FIRST_STEP
        nominal = first + step*np.arange(len(freq))
    records = []
    for name,gate in regions.items():
        rows,p = compare_cube(freq,nominal,cube,gate,on2,PS_GRID,PS_DRAWS,PS_LONG,SEED)
        for row in rows:
            records.append(dict(archive=path.name,component=name,draws=PS_DRAWS,seed=SEED,**row))
        np.savez_compressed(OUTPUT/f"{path.name}.{name}.ps.npz", **p,valid=valid,
                            ftol=FTOL,subtract_baseline=SUBTRACT_BASELINE,seed=SEED)
        fig,(a,b) = plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
        a.imshow(p["amplitudes"],aspect="auto",origin="lower",
                 extent=(-.5,gate.sum()-.5,p["grid"][0],p["grid"][-1]))
        a.set(xlabel="Index in selected bins (see NPZ bins)",ylabel="RM (rad/m²)")
        b.plot(p["grid"],p["collapsed"],label="Sum of bin amplitudes")
        b.plot(p["grid"],p["fitted_curve"],"--",label="Fitted response")
        b.set(xlabel="RM (rad/m²)",ylabel="Polarised amplitude")
        b.legend()
        fig.suptitle(f"{path.name} · {name}")
        plt.show()
        plt.close(fig)
    return records

## Run the original archive analysis

In [ ]:
if RUN_ORIGINAL:
    source = Path(INPUT).expanduser()
    if source.is_file():
        files = [source]
    elif source.is_dir():
        if not EXTENSION:
            raise ValueError("Set a filename extension for directory mode")
        files = sorted(p for p in source.iterdir() if p.is_file() and p.name.endswith(EXTENSION))
    else:
        raise FileNotFoundError(source)
    if not files:
        raise FileNotFoundError(f"No files ending in {EXTENSION!r} in {source.resolve()}")
    if GRID_POINTS < 3 or DRAWS < 2 or RM_LIMITS[0] >= RM_LIMITS[1]:
        raise ValueError("Check the RM limits, grid size and noise draw count")
    OUTPUT.mkdir(parents=True, exist_ok=True)
    results = []
    for path in files:
        try:
            rows = process(path, np.random.default_rng(SEED))
        except (RuntimeError, OSError, ValueError) as exc:
            rows = [dict(archive=path.name, status=str(exc))]
        results.extend(rows)
        print(path.name, " | ".join(row["status"] for row in rows))
    summary = OUTPUT / "component_rm.csv"
    with summary.open("w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=list(dict.fromkeys(k for row in results for k in row)))
        writer.writeheader()
        writer.writerows(results)
    print(summary.resolve())

## Run the PSRSALSA-style archive analysis
For example, with bins 80–91 and signal window 78–118, compare against `rmsynth -rm "-30 30 121" -onpulse "80 91" -onpulse2 "78 118" -bootstrap 500 -ftol 0.001 -ascii archive`. Repeat with `-long` for bin fits. Match the notebook grid, draw count, archive and preprocessing. Check the collapsed grid peak, fitted RM, FWHM, total L and RMS L before comparing uncertainty columns. With a different RNG the Monte Carlo spread itself fluctuates (roughly 1/√[2(N−1)] fractional scatter for Gaussian RM draws).

In [ ]:
if RUN_COMPARISON:
    source = Path(INPUT).expanduser()
    ps_files = [source] if source.is_file() else sorted(source.glob("*"+EXTENSION))
    if not ps_files:
        raise FileNotFoundError(f"No matching archives at {source}")
    OUTPUT.mkdir(parents=True,exist_ok=True)
    ps_rows = []
    for path in ps_files:
        try:
            ps_rows.extend(run_ps(path))
        except (ValueError,RuntimeError,OSError) as exc:
            ps_rows.append(dict(archive=path.name,status=str(exc)))
            print(path.name,exc)
    with (OUTPUT/"psrsalsa_rm.csv").open("w",newline="") as file:
        writer = csv.DictWriter(file,fieldnames=list(dict.fromkeys(k for r in ps_rows for k in r)))
        writer.writeheader()
        writer.writerows(ps_rows)
    print(OUTPUT/"psrsalsa_rm.csv")